# Day 05 — Capstone: ghép toàn bộ pipeline và xuất output cuối cùng

**Slide tham khảo chi tiết:** [day05_reference_pack.zip](_static/slides/day05_reference_pack.zip)

## Mục tiêu bài học

- ghép lại mô tả dữ liệu, hiệu năng mô hình, độ ổn định và pathway analysis trong một notebook duy nhất
- xuất thư mục output rõ ràng để dùng cho báo cáo hoặc poster
- viết được phần diễn giải ngắn gọn ngay trong notebook

## Nội dung

Day 05 không thêm kiến thức mới. Mục tiêu là tổ chức lại toàn bộ flow để biến 4 buổi trước thành một pipeline gọn, có bảng, có hình, có ghi chú, có thể mở lại để đối chiếu khi luyện thuyết trình hoặc chuẩn bị poster.

## Sản phẩm sau bài học

- `table1.csv`
- `roc_ring1.png`
- `cv_table.csv`
- `pathway_table.csv`
- `pathway_delta_median_bar.png`
- `notes_summary.txt`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
plt.rcParams["figure.figsize"] = (6,4)
sns.set_style("whitegrid")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

GITHUB_USER = "ketnoimaytinh797-dotcom"
REPO_NAME = "EGFR-Radiomics-MiniBootcamp"
BRANCH = "main"


def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out = Path(rel_path).name
    urllib.request.urlretrieve(url, out)
    return Path(out)


def load_csv(rel_path: str) -> pd.DataFrame:
    p = Path(rel_path)
    if p.exists():
        return pd.read_csv(p)
    try:
        p2 = download_from_github(rel_path)
        return pd.read_csv(p2)
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy {rel_path}. Hãy chỉnh đúng GITHUB_USER/REPO_NAME hoặc upload file thủ công.") from e


df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
ngs = load_csv("data/ngs_pathway_demo_64.csv")
out_dir = Path("outputs/day05_capstone")
out_dir.mkdir(parents=True, exist_ok=True)

## Bước 1. Tạo Table 1 gọn

In [ ]:

table1 = pd.DataFrame([
    ["n", len(df)],
    ["Age, mean ± SD", f"{df['age'].mean():.1f} ± {df['age'].std(ddof=1):.1f}"],
    ["Tumor size, median [IQR]", f"{df['tumor_size_mm'].median():.1f} [{df['tumor_size_mm'].quantile(0.25):.1f}; {df['tumor_size_mm'].quantile(0.75):.1f}]"],
    ["EGFR mutated, n (%)", f"{df['egfr_mutation'].sum()} ({df['egfr_mutation'].mean()*100:.1f}%)"]
], columns=["Variable", "Overall"])
table1


## Bước 2. Chạy mô hình ring1 và lấy ROC AUC

In [ ]:

cols = [c for c in df.columns if c.startswith("ring1_")]
X = df[cols]
y = df["egfr_mutation"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])
pipe.fit(X_train, y_train)
pred_prob = pipe.predict_proba(X_test)[:, 1]
pred_label = (pred_prob >= 0.5).astype(int)
auc = roc_auc_score(y_test, pred_prob)
auc


In [ ]:

fig, ax = plt.subplots()
fpr, tpr, _ = roc_curve(y_test, pred_prob)
ax.plot(fpr, tpr, label=f"ring1 AUC={auc:.3f}")
ax.plot([0,1], [0,1], linestyle="--", color="gray")
ax.legend()
ax.set_title("ROC — ring1 model")
fig.tight_layout()
fig.savefig(out_dir / "roc_ring1.png", dpi=150)
plt.show()


## Bước 3. Kiểm tra độ ổn định bằng 5-fold CV

In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
cv_table = pd.DataFrame({"fold": range(1, 6), "auc": cv_scores})
cv_table


## Bước 4. Tính pathway delta P trên subset NGS

In [ ]:

pipe.fit(X, y)
df["pred_prob_egfr"] = pipe.predict_proba(X)[:, 1]
merged = ngs.merge(df[["patient_id", "pred_prob_egfr"]], on="patient_id", how="left")
pathway_cols = [c for c in merged.columns if c.startswith("pathway_")]
rows = []
for pw in pathway_cols:
    g_mut = merged.loc[merged[pw] == 1, "pred_prob_egfr"].dropna()
    g_wt = merged.loc[merged[pw] == 0, "pred_prob_egfr"].dropna()
    rows.append([
        pw.replace("pathway_", ""),
        g_mut.mean() - g_wt.mean(),
        g_mut.median() - g_wt.median(),
        mannwhitneyu(g_mut, g_wt, alternative="two-sided").pvalue
    ])
pathway_table = pd.DataFrame(rows, columns=["pathway", "delta_mean_p", "delta_median_p", "p_value"]).sort_values("delta_median_p", ascending=False)
pathway_table


In [ ]:

fig, ax = plt.subplots(figsize=(7,4))
pathway_table.plot(x="pathway", y="delta_median_p", kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta median P")
ax.set_title("Pathway delta median predicted probability")
fig.tight_layout()
fig.savefig(out_dir / "pathway_delta_median.png", dpi=150)
plt.show()


## Bước 5. Xuất toàn bộ output

In [ ]:

table1.to_csv(out_dir / "table1.csv", index=False)
cv_table.to_csv(out_dir / "cv_table.csv", index=False)
pathway_table.to_csv(out_dir / "pathway_table.csv", index=False)

notes = """
- Cohort demo gồm 200 bệnh nhân, đã được tóm tắt bằng Table 1.
- Mô hình minh họa dùng ring1 cho AUC ở mức tham khảo và có thể tiếp tục đánh giá bằng CV.
- 5-fold CV cho phép nhìn độ dao động của mô hình qua nhiều cách chia dữ liệu.
- Subset NGS được dùng để tính delta mean P và delta median P theo từng pathway.
- Kết quả pathway chỉ nên diễn giải ở mức gợi ý, không khẳng định cơ chế sinh học cuối cùng.
- Toàn bộ bảng và hình đã được lưu trong outputs/day05_capstone.
""".strip()
(out_dir / "report_notes.md").write_text(notes, encoding="utf-8")
list(out_dir.iterdir())


## Tự kiểm tra cuối chương trình

1. Trong toàn bộ flow 5 buổi, bước nào là phần mô tả, bước nào là phần suy luận, bước nào là phần mô hình hóa?  
2. Nếu phải trình bày ngắn gọn 1 phút, nên đi theo thứ tự nào?  
3. Output tối thiểu nào cần mang theo để đối chiếu khi luyện phỏng vấn?  
4. Nếu chuyển sang dữ liệu thật, phần nào cần kiểm tra đầu tiên?

## Kết thúc mini bootcamp

Sau Day 05, học sinh đã có thể đọc được toàn bộ flow code của báo cáo ở mức vừa đủ để tự học trên Colab và trao đổi lại các điểm chưa rõ.